# AP2: Figurenauswertung

Dieses Notebook belegt die Werte, die `kapitel7.ipynb` für das Figurenclustering fest einträgt.

## Vorbereitung

### Arbeitsverzeichnis

In [ ]:
#Projekt-Stammverzeichnis suchen (Ordner mit "Korpus/" oder dem Export)
import os
from pathlib import Path

PROJEKT = Path.cwd()
for _kandidat in [PROJEKT, *PROJEKT.parents]:
    if (_kandidat / "prototyp_ergebnisse_alle.json").is_file():
        PROJEKT = _kandidat
        break
os.chdir(PROJEKT)
print("Arbeitsverzeichnis:", PROJEKT)

#Eingaben: Export der Pipeline und Clusterkarte aus ap1.ipynb
P_EXPORT = Path("prototyp_ergebnisse_alle.json")
P_CLUSTERKARTE = Path("ergebnisse_kapitel7_ap1") / "clusterkarte.csv"
for pfad in (P_EXPORT, P_CLUSTERKARTE):
    print("vorhanden" if pfad.exists() else "FEHLT    ", "|", pfad)
if not P_CLUSTERKARTE.exists():
    raise FileNotFoundError("Clusterkarte fehlt: zuerst kap7_ap1.ipynb ausführen")

Arbeitsverzeichnis: /home/timc/Workspace/fairytale-personality-profiling/Abgabe
vorhanden | prototyp_ergebnisse_alle.json
vorhanden | ergebnisse_kapitel7_ap1/clusterkarte.csv


### Einstellungen

In [ ]:
import json

import numpy as np
import pandas as pd
from scipy.stats import chi2

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 250)

#Schwellen der Figurenauswertung (gesetzt; die Kalibrierung misst sie nach)
MIN_BELEGE_ZELLE = 3                #Test einer Zelle aus Rolle und Merkmal (Konvention aus AP1)
MIN_BELEGE_PROFIL = 5               #Rolle in der Merkmalsauswertung (Konvention aus AP1)
MIN_BELEGE_VEKTOR = 10              #Rolle im Profilvergleich
MIN_BELEGE_VEKTOR_SENSITIV = 5      #Variante für den bereinigten Rand-Index, gleich der Profilschwelle
MIN_BELEGE_EBENENWAHL = 3           #Rollen im Vergleich der Merkmalsebenen (AP8b)
FDR_ALPHA = 0.05                    #Fehlerkontrolle nach Benjamini-Hochberg
TOP_JE_FIGUR = 8                    #Berichtstiefe: stärkste Merkmale je Rolle

#Zerlegung der Rollen (gesetzt): Clusterzahl ab K_MIN, größtes Cluster höchstens 40 Prozent
K_MIN = 4
MAX_ANTEIL_GROESSTES = 0.40

#Rollenpaare (gesetzt, Analysebeispiele)
PAARE = [("König", "Königin"), ("Prinz", "Prinzessin"), ("Stiefmutter", "Mutter")]

#Genus aus den Koreferenzprotokollen (gesetzt; angewandt in kapitel7.ipynb)
PRONOMEN_MASKULIN = {"er", "ihn"}   #"ihm" entfällt, weil es auch zum Neutrum gehört
PRONOMEN_FEMININ = {"sie"}
GENUS_MIN_PRONOMEN = 5              #Mindestzahl gewerteter Pronomen je Rolle
GENUS_DOMINANZ = 0.80               #Anteil, ab dem eine Richtung entschieden ist
GENUS_MAX_PLURALANTEIL = 0.50       #Rollen darüber gelten als plural oder kollektiv
GENUS_CLUSTER_EINHEIT = 0.80        #Koreferenzcluster mit gemischten Pronomen werden verworfen
GENUS_MIN_ERWAEHNUNGEN = 2          #Gutschrift nur bei mindestens zwei Erwähnungen im Cluster

#Tragfähigkeit je Sammlung: Verdachtsquoten aus AP4a (zitiert, nicht im Export) und Schwellen der Einstufung
VERDACHTSQUOTE_AP4A = {"Japanische Märchen": 0.060, "Brüder Grimm": 0.125,
                       "H. C. Andersen": 0.213, "Arabische Märchen": 0.368}
TRAGFAEHIG_BIS = 0.15
EINGESCHRAENKT_BIS = 0.25

#Wirkung der Schwellen: geprüftes Raster
SCHWELLEN_RASTER = (1, 2, 3, 5, 8, 10, 15, 20, 30, 50)

#Kalibrierung: gesetzt sind die Kriterien, die Empfehlung folgt daraus
KALIBRIER_STARTWERT = 8
KALIBRIER_PERMUTATIONEN = 20                #Vertauschungen der Rollenetiketten
ZELLSCHWELLE_RASTER = (1, 2, 3, 4, 5, 6)
KALIBRIER_FEHLTREFFER_MAX = FDR_ALPHA       #Anteil falscher Signifikanzen höchstens auf dem Niveau der Korrektur
KALIBRIER_WIEDERHOLUNGEN = 25               #zufällige Halbierungen je Rolle
BELEGKLASSEN = ((4, 5), (6, 7), (8, 9), (10, 14), (15, 19), (20, 29), (30, 49), (50, 10 ** 6))
KALIBRIER_VEKTOR_MIN = 0.60                 #mittlerer Kosinus der Hälften
KALIBRIER_PROFIL_MIN = 0.50                 #mittlere Übereinstimmung der drei stärksten Merkmale
KALIBRIER_MIN_ROLLEN_JE_KLASSE = 5          #Belegklasse zählt erst ab fünf Rollen

#Wahl der Merkmalsebene (AP8b): gesetzt ist die Halbierungsprobe, die Ebene folgt daraus
EBENEN_STARTWERT = 8
EBENEN_HALBIERUNGEN = 20
EBENEN_MIN_BELEGE_PROBE = 6                 #beide Hälften behalten mindestens drei Belege


def prozent(teil, ganzes):
    #Anteil in Prozent mit einer Nachkommastelle
    return round(100 * teil / ganzes, 1)

### Belegtabelle und Clusterkarte

In [ ]:
with open(P_EXPORT, encoding="utf-8") as f:
    daten = json.load(f)
print("Pipeline-Version:", daten.get("meta", {}).get("notebook_version"))

#Belegtabelle: eine Zeile je zugeordnetem Beleg mit Rolle, Normform, Herkunft und Instanzverdacht
zeilen = []
for z in daten["adjektiv_zuordnungen_alle"]:
    if z.get("status") != "zugeordnet" or not z.get("figur"):
        continue
    normform = (z.get("lemma_norm") or z.get("lemma") or "").strip().lower()
    if not normform:
        continue
    zeilen.append({"figur": z["figur"], "merkmal_normform": normform, "sammlung": z.get("sammlung", "(Wurzel)"),
                   "maerchen": z.get("maerchen"), "instanz_verdacht": bool(z.get("instanz_verdacht"))})
df_belege = pd.DataFrame(zeilen)

#Clusterebene: Name des Clusters aus ap1; außerhalb der Karte bleibt das Feld leer
karte = pd.read_csv(P_CLUSTERKARTE, dtype=str, keep_default_na=False)
NAME_JE_FORM = dict(zip(karte["normform"].str.strip().str.lower(), karte["name"]))
df_belege["merkmal_cluster"] = df_belege["merkmal_normform"].map(NAME_JE_FORM)
MERKMALSEBENEN = {"merkmal_normform": "Normform", "merkmal_cluster": "Cluster"}

mit_cluster = int(df_belege["merkmal_cluster"].notna().sum())
print("Zugeordnete Belege:", len(df_belege), "| Rollen:", df_belege["figur"].nunique(),
      "| Normformen:", df_belege["merkmal_normform"].nunique())
print("Clusterkarte:", len(NAME_JE_FORM), "Normformen in", karte["name"].nunique(), "Clustern")
print("Belege auf der Clusterebene:", mit_cluster, "|", prozent(mit_cluster, len(df_belege)), "%")

Pipeline-Version: V1.0
Zugeordnete Belege: 4254 | Rollen: 550 | Normformen: 901
Clusterkarte: 380 Normformen in 59 Clustern
Belege auf der Clusterebene: 3712 | 87.3 %


## 1 Tragfähigkeit je Sammlung

Verdachtsquote der Instanzambiguität, gemessen und aus AP4a zitiert. Maßgeblich ist die höhere Quote.

In [5]:
def einstufung(quote):
    #Tragfähigkeit sammlungsinterner Aussagen über Rollen (Bezeichnungen wie in Kapitel 7)
    if quote < TRAGFAEHIG_BIS:
        return "tragfähig"
    if quote < EINGESCHRAENKT_BIS:
        return "eingeschränkt tragfähig"
    return "nicht belastbar"


#Gemessen: Anteil der Belege mit Instanzverdacht; zitiert: AP4a; maßgeblich ist die höhere Quote
df_tragfaehigkeit = (df_belege.groupby("sammlung")
                     .agg(belege=("figur", "size"), rollen=("figur", "nunique"), maerchen=("maerchen", "nunique"),
                          gemessen=("instanz_verdacht", "mean"))
                     .reset_index())
df_tragfaehigkeit["zitiert_ap4a"] = df_tragfaehigkeit["sammlung"].map(VERDACHTSQUOTE_AP4A)
df_tragfaehigkeit["massgeblich"] = df_tragfaehigkeit[["gemessen", "zitiert_ap4a"]].max(axis=1)
df_tragfaehigkeit["einstufung_gemessen"] = df_tragfaehigkeit["gemessen"].map(einstufung)
df_tragfaehigkeit["einstufung"] = df_tragfaehigkeit["massgeblich"].map(einstufung)
df_tragfaehigkeit = df_tragfaehigkeit.sort_values("massgeblich").reset_index(drop=True)
print("Instanzverdacht korpusweit:", prozent(int(df_belege["instanz_verdacht"].sum()), len(df_belege)), "%")
for spalte in ("gemessen", "zitiert_ap4a", "massgeblich"):
    df_tragfaehigkeit[spalte] = (100 * df_tragfaehigkeit[spalte]).round(1)
df_tragfaehigkeit

Instanzverdacht korpusweit: 15.5 %


,sammlung,belege,rollen,maerchen,gemessen,zitiert_ap4a,massgeblich,einstufung_gemessen,einstufung
0,Japanische Märchen,552,127,16,6.0,6.0,6.0,tragfähig,tragfähig
1,Brüder Grimm,1543,233,45,11.9,12.5,12.5,tragfähig,tragfähig
2,H. C. Andersen,1056,172,11,12.8,21.3,21.3,tragfähig,eingeschränkt tragfähig
3,Arabische Märchen,1103,168,8,27.8,36.8,36.8,nicht belastbar,nicht belastbar


## 2 Wirkung der Schwellen

Rollen und Belegmasse je Mindestzahl der Belege, getrennt nach Merkmalsebene.

In [6]:
#Rollen und Belegmasse je Schwelle auf beiden Merkmalsebenen; markiert sind die eingesetzten Schwellen
IM_EINSATZ = {MIN_BELEGE_ZELLE: "Zellschwelle", MIN_BELEGE_PROFIL: "Profilschwelle", MIN_BELEGE_VEKTOR: "Vektorschwelle"}
zeilen = []
for spalte, ebene in MERKMALSEBENEN.items():
    teil = df_belege.dropna(subset=[spalte])
    groesse = teil.groupby("figur").size()
    typen = teil.groupby("figur")[spalte].nunique()
    for schwelle in SCHWELLEN_RASTER:
        bleibt = groesse[groesse >= schwelle]
        zeilen.append({"Ebene": ebene, "Schwelle": schwelle, "Rollen": len(bleibt),
                       "Rollen %": prozent(len(bleibt), len(groesse)), "Belege": int(bleibt.sum()),
                       "Belege %": prozent(int(bleibt.sum()), int(groesse.sum())), "Median": float(bleibt.median()),
                       "Rollen nach Merkmalszahl": int((typen >= schwelle).sum()),
                       "eingesetzt": IM_EINSATZ.get(schwelle, "")})
df_schwellen = pd.DataFrame(zeilen)
df_schwellen

,Ebene,Schwelle,Rollen,Rollen %,Belege,Belege %,Median,Rollen nach Merkmalszahl,eingesetzt
0,Normform,1,550,100.0,4254,100.0,2.0,550,
1,Normform,2,325,59.1,4029,94.7,5.0,304,
2,Normform,3,232,42.2,3843,90.3,7.5,214,Zellschwelle
3,Normform,5,164,29.8,3613,84.9,11.5,148,Profilschwelle
4,Normform,8,116,21.1,3330,78.3,17.5,98,
5,Normform,10,96,17.5,3165,74.4,20.0,77,Vektorschwelle
6,Normform,15,66,12.0,2812,66.1,31.0,41,
7,Normform,20,51,9.3,2559,60.2,36.0,30,
8,Normform,30,34,6.2,2170,51.0,45.0,16,
9,Normform,50,14,2.5,1398,32.9,69.0,6,


## 3 Kalibrierung der Schwellen

Zellschwelle: Rollenetiketten 20-mal vertauschen. Der Anteil falscher Signifikanzen darf das Niveau der Korrektur nicht übersteigen.

In [7]:
def bh_fdr(p_werte, alpha=FDR_ALPHA):
    """Benjamini-Hochberg: Signifikanzentscheid je Test."""
    p = np.asarray(p_werte, dtype=float)
    m = len(p)
    if m == 0:
        return np.array([], dtype=bool)
    ordnung = np.argsort(p)
    korrigiert = p[ordnung] * m / np.arange(1, m + 1)
    korrigiert = np.clip(np.minimum.accumulate(korrigiert[::-1])[::-1], 0, 1)
    p_fdr = np.empty(m)
    p_fdr[ordnung] = korrigiert
    return p_fdr <= alpha


def g2_matrix(a):
    """G² (Dunning 1993) für alle Zellen einer Kreuztabelle Rolle x Merkmal."""
    R = a.sum(axis=1, keepdims=True)
    C = a.sum(axis=0, keepdims=True)
    N = a.sum()
    b, c, d = C - a, R - a, N - R - C + a
    Ea, Eb = R * C / N, (N - R) * C / N
    Ec, Ed = R * (N - C) / N, (N - R) * (N - C) / N

    def term(x, e):
        with np.errstate(divide="ignore", invalid="ignore"):
            t = x * np.log(x / e)
        return np.where(x > 0, t, 0.0)

    return np.clip(2 * (term(a, Ea) + term(b, Eb) + term(c, Ec) + term(d, Ed)), 0, None)


def zellen_test(df, ebene, min_zelle, min_gruppe=MIN_BELEGE_PROFIL):
    """Geprüfte und signifikante Zellen: Rollen ab min_gruppe Belegen, Zellen ab min_zelle Belegen."""
    teil = df.dropna(subset=[ebene])
    groesse = teil.groupby("figur").size()
    teil = teil[teil["figur"].isin(groesse[groesse >= min_gruppe].index)]
    a = pd.crosstab(teil["figur"], teil[ebene]).values.astype(float)
    getestet = a >= min_zelle
    signifikant = bh_fdr(chi2.sf(g2_matrix(a)[getestet], df=1))
    return int(getestet.sum()), int(signifikant.sum())


#Nullkalibrierung auf der Clusterebene: Rollenetiketten vertauschen, Randverteilungen bleiben erhalten
EBENE = "merkmal_cluster"
zufall = np.random.default_rng(KALIBRIER_STARTWERT)
beobachtet = {s: zellen_test(df_belege, EBENE, s) for s in ZELLSCHWELLE_RASTER}
fehltreffer = {s: [] for s in ZELLSCHWELLE_RASTER}
df_null = df_belege.copy()
for _ in range(KALIBRIER_PERMUTATIONEN):
    df_null["figur"] = zufall.permutation(df_belege["figur"].values)
    for s in ZELLSCHWELLE_RASTER:
        geprueft, treffer = zellen_test(df_null, EBENE, s)
        fehltreffer[s].append(treffer / geprueft if geprueft else np.nan)

df_zellschwelle = pd.DataFrame([{"Zellschwelle": s, "geprüfte Zellen": beobachtet[s][0],
                                 "signifikant": beobachtet[s][1],
                                 "Fehltreffer": float(np.nanmean(fehltreffer[s])),
                                 "höchstens": float(np.nanmax(fehltreffer[s]))} for s in ZELLSCHWELLE_RASTER])
df_zellschwelle["hält"] = df_zellschwelle["Fehltreffer"] <= KALIBRIER_FEHLTREFFER_MAX
zulaessig = df_zellschwelle[df_zellschwelle["hält"]]
EMPFEHLUNG_ZELLE = int(zulaessig["Zellschwelle"].min()) if len(zulaessig) else None
print("Kleinste Zellschwelle mit höchstens", prozent(KALIBRIER_FEHLTREFFER_MAX, 1), "% Fehltreffern:", EMPFEHLUNG_ZELLE,
      "| eingesetzt:", MIN_BELEGE_ZELLE)
df_zellschwelle.round(5)

Kleinste Zellschwelle mit höchstens 5.0 % Fehltreffern: 1 | eingesetzt: 3


,Zellschwelle,geprüfte Zellen,signifikant,Fehltreffer,höchstens,hält
0,1,1466,117,0.00010,0.00104,True
1,2,555,202,0.00193,0.00716,True
2,3,270,153,0.00668,0.02979,True
3,4,166,101,0.00835,0.03252,True
4,5,98,65,0.00804,0.03846,True
5,6,67,50,0.00523,0.02381,True


Profil- und Vektorschwelle: Belege jeder Rolle 25-mal zufällig halbieren. Die Zelle setzt den Zufallsgenerator der vorigen Zelle fort und wird deshalb direkt nach ihr ausgeführt.

In [8]:
def halbierung(codes, anzahl_merkmale, wiederholungen):
    """Kosinus und Übereinstimmung der drei stärksten Merkmale zweier zufälliger Hälften einer Rolle."""
    kosinus, jaccard = [], []
    haelfte = len(codes) // 2
    for _ in range(wiederholungen):
        misch = zufall.permutation(codes)
        a = np.bincount(misch[:haelfte], minlength=anzahl_merkmale)
        b = np.bincount(misch[haelfte:2 * haelfte], minlength=anzahl_merkmale)
        norm = np.linalg.norm(a) * np.linalg.norm(b)
        kosinus.append(float(a @ b / norm) if norm else np.nan)
        #Bei Gleichstand entscheidet die Reihenfolge des Merkmalsindex
        spitze_a = set(np.argsort(-a)[:3][a[np.argsort(-a)[:3]] > 0])
        spitze_b = set(np.argsort(-b)[:3][b[np.argsort(-b)[:3]] > 0])
        vereinigung = spitze_a | spitze_b
        jaccard.append(len(spitze_a & spitze_b) / len(vereinigung) if vereinigung else np.nan)
    return float(np.nanmean(kosinus)), float(np.nanmean(jaccard))


def belegklasse(n):
    #Klasse und ihre Untergrenze
    for unten, oben in BELEGKLASSEN:
        if unten <= n <= oben:
            return (str(unten) + "–" + str(oben) if oben < 10 ** 6 else "ab " + str(unten)), unten
    return None, np.nan


#Verlässlichkeit auf der Clusterebene; derselbe Zufallsgenerator läuft nach den Vertauschungen weiter
teil = df_belege.dropna(subset=[EBENE])
merkmal_index = {m: i for i, m in enumerate(sorted(teil[EBENE].unique()))}
zeilen = []
for rolle, gruppe in teil.groupby("figur"):
    if len(gruppe) < BELEGKLASSEN[0][0]:
        continue
    kos, jac = halbierung(gruppe[EBENE].map(merkmal_index).to_numpy(), len(merkmal_index), KALIBRIER_WIEDERHOLUNGEN)
    klasse, unten = belegklasse(len(gruppe))
    zeilen.append({"figur": rolle, "belege": len(gruppe), "belegklasse": klasse, "untergrenze": unten,
                   "kosinus": kos, "spitzen_jaccard": jac})
df_reliabilitaet = pd.DataFrame(zeilen)
df_klassen = (df_reliabilitaet.groupby("belegklasse")
              .agg(rollen=("figur", "size"), untergrenze=("untergrenze", "first"),
                   kosinus=("kosinus", "mean"), spitzen_jaccard=("spitzen_jaccard", "mean"))
              .reset_index().sort_values("untergrenze").reset_index(drop=True))
df_klassen["besetzt"] = df_klassen["rollen"] >= KALIBRIER_MIN_ROLLEN_JE_KLASSE

#Empfehlung: kleinste besetzte Klasse, die das Kriterium erreicht
besetzt = df_klassen[df_klassen["besetzt"]]
vektor = besetzt[besetzt["kosinus"] >= KALIBRIER_VEKTOR_MIN]
profil = besetzt[besetzt["spitzen_jaccard"] >= KALIBRIER_PROFIL_MIN]
EMPFEHLUNG_VEKTOR = int(vektor["untergrenze"].min()) if len(vektor) else None
EMPFEHLUNG_PROFIL = int(profil["untergrenze"].min()) if len(profil) else None
print("Vektorschwelle: Kosinus ab", KALIBRIER_VEKTOR_MIN, "erreicht ab", EMPFEHLUNG_VEKTOR, "Belegen | eingesetzt:",
      MIN_BELEGE_VEKTOR)
print("Profilschwelle: Spitzenübereinstimmung ab", KALIBRIER_PROFIL_MIN, "erreicht ab", EMPFEHLUNG_PROFIL,
      "Belegen | eingesetzt:", MIN_BELEGE_PROFIL)
df_klassen.round(4)

Vektorschwelle: Kosinus ab 0.6 erreicht ab 30 Belegen | eingesetzt: 10
Profilschwelle: Spitzenübereinstimmung ab 0.5 erreicht ab None Belegen | eingesetzt: 5


,belegklasse,rollen,untergrenze,kosinus,spitzen_jaccard,besetzt
0,4–5,41,4,0.3023,0.2369,True
1,6–7,26,6,0.3045,0.2071,True
2,8–9,18,8,0.3945,0.2078,True
3,10–14,29,10,0.3864,0.2043,True
4,15–19,12,15,0.4654,0.2289,True
5,20–29,16,20,0.4823,0.2202,True
6,30–49,18,30,0.6333,0.2671,True
7,ab 50,11,50,0.8324,0.3493,True


## 4 Wahl der Merkmalsebene (AP8b)

Normform gegen Cluster: Reichweite, Dichte und Wiedererkennung zweier zufälliger Hälften (20 Ziehungen, Rollen ab sechs Belegen).

In [9]:
def baue_vektoren(df, ebene, min_belege):
    """Längennormierte Anteilsvektoren (Rolle x Merkmal) der Rollen ab min_belege Belegen."""
    teil_ = df.dropna(subset=[ebene])
    groesse = teil_.groupby("figur").size()
    teil_ = teil_[teil_["figur"].isin(groesse[groesse >= min_belege].index)]
    matrix = pd.crosstab(teil_["figur"], teil_[ebene]).astype(float)
    matrix = matrix.div(matrix.sum(axis=1), axis=0)
    return matrix.div(np.linalg.norm(matrix.values, axis=1), axis=0), teil_


def wiedererkennung(teil_, ebene, generator):
    """Mittlerer Kosinus zweier zufälliger Hälften je Rolle (Rollen ab EBENEN_MIN_BELEGE_PROBE Belegen)."""
    schluessel = {m: i for i, m in enumerate(sorted(teil_[ebene].unique()))}
    werte = []
    for _, gruppe in teil_.groupby("figur"):
        if len(gruppe) < EBENEN_MIN_BELEGE_PROBE:
            continue
        codes = gruppe[ebene].map(schluessel).to_numpy()
        haelfte = len(codes) // 2
        einzeln = []
        for _ in range(EBENEN_HALBIERUNGEN):
            misch = generator.permutation(codes)
            a = np.bincount(misch[:haelfte], minlength=len(schluessel))
            b = np.bincount(misch[haelfte:2 * haelfte], minlength=len(schluessel))
            norm = np.linalg.norm(a) * np.linalg.norm(b)
            einzeln.append(float(a @ b / norm) if norm else 0.0)
        werte.append(float(np.mean(einzeln)))
    return float(np.mean(werte)), len(werte)


#Beide Ebenen mit eigenem Zufallsgenerator, erst die Normform-, dann die Clusterebene
generator = np.random.default_rng(EBENEN_STARTWERT)
zeilen = []
for spalte, ebene in MERKMALSEBENEN.items():
    matrix, teil_ebene = baue_vektoren(df_belege, spalte, MIN_BELEGE_EBENENWAHL)
    werte = matrix.values @ matrix.values.T
    paare = werte[np.triu_indices_from(werte, k=1)]
    wert, geprueft = wiedererkennung(teil_ebene, spalte, generator)
    zeilen.append({"Ebene": ebene, "Rollen": matrix.shape[0], "Merkmale": matrix.shape[1],
                   "Belege": len(teil_ebene), "Belege %": prozent(len(teil_ebene), len(df_belege)),
                   "Paare mit gemeinsamem Merkmal %": prozent(int((paare > 0.001).sum()), len(paare)),
                   "Median": float(np.median(paare)), "Wiedererkennung": wert, "geprüfte Rollen": geprueft})
df_ebenen = pd.DataFrame(zeilen)

#Ausschlaggebend ist die Wiedererkennung der Profile
MERKMALSEBENE = df_ebenen.loc[df_ebenen["Wiedererkennung"].idxmax(), "Ebene"]
print("Gewählte Merkmalsebene für den Profilvergleich:", MERKMALSEBENE)
df_ebenen.round(3)

Gewählte Merkmalsebene für den Profilvergleich: Cluster


,Ebene,Rollen,Merkmale,Belege,Belege %,Paare mit gemeinsamem Merkmal %,Median,Wiedererkennung,geprüfte Rollen
0,Normform,232,826,3843,90.3,31.8,0.000,0.328,146
1,Cluster,208,59,3338,78.5,61.2,0.076,0.463,130
